# Tutorial: enumerating metabolites with Metabolic Forest

This notebook shows how to use `xenosite.metabolite` to generate metabolite structures and search pathways between a reactant and a product.

Site-of-metabolism *scores* are available at [xenosite.org](https://xenosite.org). This package enumerates *structures*.

Please cite Hughes et al., *Metabolic Forest*, *J. Chem. Inf. Model.* 2020, DOI [10.1021/acs.jcim.0c00360](https://doi.org/10.1021/acs.jcim.0c00360) if you use this software. Copy-paste BibTeX is in the [README](../README.md#citation). Rulesets and the papers they match (Rainbow Phase I, quinone, bioactivation) are in [docs/rulesets.md](../docs/rulesets.md).

In [ ]:
from rdkit import Chem
from xenosite.metabolite import bfs, rules, RuleSet, PhaseOneRS, load_ruleset

## Enumerate metabolites of one rule

Hydroxylate propane and print the product SMILES.

In [ ]:
mol = Chem.MolFromSmiles("CCC")
for site, products in rules.Hydroxylation().metabolites(mol):
    print(site, [Chem.MolToSmiles(p) for p in products])

## Combine rules

`RuleSet` groups rules. `PhaseOneRS` is the Phase I collection used in Metabolic Forest.

In [ ]:
print(sorted({rule.name for rule in PhaseOneRS}))

combo = RuleSet([rules.Epoxidation(), rules.EpoxideOpening()], name="epoxide")
reactant = Chem.MolFromSmiles("c1ccccc1")
product = Chem.MolFromSmiles("C1=CC=CC(O)C1O")
smiles, steps, mols = next(combo.find_path(reactant, product, depth=2))
print(smiles)
print(steps)

## Search a pathway with `bfs`

Pass reactant and product SMILES. Use `phase1=True` for Phase I site strings.

In [ ]:
smiles, steps, mols = next(bfs(["CCO", "CC=O"], ruleset="PhaseOneRS", phase1=True))
print(smiles)
print(steps)

## Optional: expand a metabolite network

Install the extra with `uv add "xenosite-metabolite[network]"`.

In [ ]:
try:
    from xenosite.metabolite.net import MetaboliteNetwork
except ImportError:
    print("Install the network extra to run this cell.")
else:
    net = MetaboliteNetwork("CCO")
    net.expand(load_ruleset("PhaseOneRS"))
    for path in net.paths("CCO", "CC=O"):
        print(path)
        break